# Attention-based interpretation

Reproduce the atom-level saliency and gene-level cross-attention plots from the paper.

Fill in the placeholders below; the rest is self-contained.

In [ ]:
# === Fill these in ===
CHECKPOINT_PATH = 'path/to/your/farm_checkpoint.h5'
DATA_ROOT       = 'path/to/FARM_dataset_v1'

# Drugs to inspect at the atom level (any subset of the 36 training drugs)
DRUGS_TO_PLOT = ['ampicillin', 'ofloxacin', 'tetracycline']

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from farm import predict, data as farm_data

drug_table = pd.read_csv(os.path.join(
    DATA_ROOT, 'reference', 'drug_smiles_36training.tsv'), sep='\t')
drug_to_smiles = dict(zip(drug_table.drug, drug_table.SMILES))

## A. Atom-level saliency (Figure 3 style)

For each drug, request the model's atom-level saliency vector and overlay it on the 2-D molecular graph.

In [ ]:
def overlay_saliency(smiles, saliency, ax):
    mol = Chem.MolFromSmiles(smiles)
    n_atoms = mol.GetNumAtoms()
    saliency = np.asarray(saliency).ravel()[:n_atoms]
    if saliency.max() > 0:
        saliency = saliency / saliency.max()
    highlights = {i: (1.0, 1 - s, 1 - s) for i, s in enumerate(saliency)}
    drawer = rdMolDraw2D.MolDraw2DCairo(400, 300)
    rdMolDraw2D.PrepareAndDrawMolecule(
        drawer, mol,
        highlightAtoms=list(range(n_atoms)),
        highlightAtomColors=highlights,
    )
    drawer.FinishDrawing()
    import io
    from PIL import Image
    img = Image.open(io.BytesIO(drawer.GetDrawingText()))
    ax.imshow(img); ax.axis('off')

fig, axes = plt.subplots(1, len(DRUGS_TO_PLOT), figsize=(4 * len(DRUGS_TO_PLOT), 3))
if len(DRUGS_TO_PLOT) == 1: axes = [axes]
for ax, drug in zip(axes, DRUGS_TO_PLOT):
    smi = drug_to_smiles[drug]
    # We pass an arbitrary isolate just to satisfy the model; we only
    # care about the drug-side saliency, which is invariant to the gene branch.
    gene_vec = np.zeros(1962, dtype=np.uint8)
    r = predict.predict_one(smi, gene_vec, gene_vec,
                            checkpoint=CHECKPOINT_PATH,
                            return_attention=True)
    if 'atom_saliency' in r:
        overlay_saliency(smi, r['atom_saliency'], ax)
        ax.set_title(drug)
    else:
        ax.text(0.5, 0.5, 'no saliency in checkpoint', ha='center')
plt.tight_layout(); plt.show()

## B. Gene-level cross-attention (Figure 4 style)

For one drug, plot the top-50 KEGG-ortholog (KO) clusters by attention score and annotate the named resistance genes that show up in the paper.

In [ ]:
DRUG = 'doripenem'  # try doripenem, amikacin, tetracycline, …

kegg = pd.read_csv(os.path.join(DATA_ROOT, 'reference', 'KEGG_orthology.txt'),
                   sep='\t', header=None, names=['KO_id', 'annotation'])
ko_annot = dict(zip(kegg.KO_id, kegg.annotation))

gene_vec = np.zeros(1962, dtype=np.uint8)
r = predict.predict_one(drug_to_smiles[DRUG], gene_vec, gene_vec,
                        checkpoint=CHECKPOINT_PATH, return_attention=True)

if 'gene_attention' not in r:
    print('Checkpoint did not expose gene_attention. Hook the model.layers '
          'directly to extract it.')
else:
    scores = np.asarray(r['gene_attention']).ravel()
    top50 = np.argsort(scores)[::-1][:50]
    df = pd.DataFrame({
        'rank': range(1, 51),
        'gene_cluster_idx': top50,
        'attention_score': scores[top50],
    })
    print(df.head(10))
    plt.figure(figsize=(8, 6))
    plt.barh(df['rank'], df['attention_score'])
    plt.gca().invert_yaxis()
    plt.xlabel('Cross-attention score')
    plt.ylabel(f'Top-50 attended KO-cluster (rank) — {DRUG}')
    plt.tight_layout(); plt.show()